# Setup

In [2]:
from import_libraries import *
import global_variables as gb_l
import libraries.common_libraries.generic_library as cm_l
import libraries.common_libraries.geom_library as geom_l
from libraries.lead.mlc_config import GeoMLClassifier as LeadConfig
from libraries.lead.cleanup.lead_question_template import LeadTemplate

In [ ]:
if "project" not in dir():
    print(">>>> STANDALONE RUN <<<<")
    project = LeadConfig(customer="Dudley", prediction_year=2024)
    project.ban_year = 1985 # from onboarding document Jan 1,1986
    project.customer_year_bins = [0, 1929, 1950, 1979, project.ban_year, project.prediction_year]
else:
    print(">>>> PIPELINE RUN <<<<")
    
cm_l.delete_folder(project.work_dir)
cm_l.create_folder(project.work_dir)

# Get input data

In [ ]:
data_files = {'service_pt': {'file': os.path.join(project.data_dir, "pipes", "ServicePoints_20231121.gdb"), 'layer':'ServicePoints'}}

cm_l.add_data_files(project, project.data_dir, data_files)

In [ ]:
service_pt = project.service_pt

In [ ]:
pipe_id_field = project.field_mapping['pipe_id_field']
diameter_field = project.field_mapping['diameter_field']
material_field = project.field_mapping['material_field']
install_yr_field = project.field_mapping['install_yr_field']
verified_field = project.field_mapping['verified_field']
vrsource_field = project.field_mapping['vrsource_field']
active_field = project.field_mapping['active_field']
abandoned_field = project.field_mapping['abandoned_field']
abandon_yr_field = project.field_mapping['abandon_yr_field']
address_field = project.field_mapping['address_field']
join_public_field = project.field_mapping['join_public_field']
join_private_field = project.field_mapping['join_private_field']

# Custom cleanup

In [ ]:
service_pt.info()

## Standardize columns

In [ ]:
service_pt.drop(columns=['address'], inplace=True)
srv_field_mapping = {'LocationalIdentifer': pipe_id_field,
                     'SiteID': address_field,
                     'PublicSLSize': 'pub_diameter',
                     'PrivateSLSize': 'pr_diameter',
                     'Public_SLInstallDate': 'pub_install_dt',
                     'Private_SLInstallDate': 'pr_install_dt',
                     'CurrentPublicSLMaterial': 'pub_material',
                     'CurrentPrivateSLMaterial': 'pr_material',
                     'Public_Verification': 'pub_verify',
                     'Private_Verification': 'pr_verify',
                     'CAMAYearBuilt': 'bld_year',
                     'CAMALUCode': 'land_use',
                     'LSLReplacementDate': 'replaced_dt',
                     'Active_Inactive': active_field,  # according to the customer all service lines should be considered as active
                     'EntireServiceLine': 'sl_label', # GRR, LEAD, UNKNOWN, NO-LEAD,
                     'ParcelIdentifier': 'parcel_id', # parcels ??
                     'Status': 'status',  # M ot T or U ??
                     'geometry': 'geometry'}
service_pt = cm_l.standardize_fields(service_pt, srv_field_mapping)

In [ ]:
for col in ['pr_install_dt', 'bld_year']:
    service_pt[col] = service_pt[col].fillna(0).astype(int)
cond_update_pr_install = (service_pt['pr_install_dt']==0) & (service_pt['bld_year']>0)
service_pt.loc[cond_update_pr_install, 'pr_install_dt'] = service_pt[cond_update_pr_install]['bld_year']

## Split public, private, active, abandoned pipes

In [ ]:
pub_pipes = service_pt[[pipe_id_field, address_field, 'pub_diameter', 'pub_install_dt', 'pub_material', 'pub_verify', active_field, 'geometry']].copy()
pr_pipes = service_pt[[pipe_id_field, 'pr_diameter', 'pr_install_dt', 'pr_material', 'pr_verify', active_field, 'geometry']].copy()

pub_pipes = pub_pipes.rename(columns={'pub_diameter': diameter_field,
                                      'pub_install_dt': install_yr_field,
                                      'pub_material': material_field,
                                      'pub_verify': verified_field})

pr_pipes = pr_pipes.rename(columns={'pr_diameter': diameter_field,
                                    'pr_install_dt': install_yr_field,
                                    'pr_material': material_field,
                                    'pr_verify': verified_field})

## Define city, zip, state, coutry

In [ ]:
# TODO: in case the address is included in the columns, then set here the city, zip, state, country
# the address field must not contain the city and the zip, must be in the form 'Street number, Street name', like '50, Main Street'
for df in [pub_pipes, pr_pipes]:
    if address_field in list(df.columns):
        df['city'] = "DUDLEY"
        df['zip'] = "01571" # if it is not known then set df['zip'] = None
        df['state'] = "MA"
        df['country'] = "USA"

## Define diameter dictionary

In [ ]:
pub_pipes[diameter_field].value_counts(dropna=False)

In [ ]:
pr_pipes[diameter_field].value_counts(dropna=False)

In [ ]:
# TODO: create a dictionary with diameter mappings; it the original diameter value contains double quotes then do not include it in the mappings
match_diameters = {'3/4': 0.75,
                   '1 1/2': 1.5,
                   '1/2': 0.5,
                   'OTH': 0}    

## Define material dictionary

In [ ]:
# check the materials and ask 1.the sales team and 2. the customer, if there are any peculiar materials

In [ ]:
pub_pipes[material_field].value_counts(dropna=False)

In [ ]:
pr_pipes[material_field].value_counts(dropna=False)

In [ ]:
# TODO: create a dictionary with material mappings

## must be in upper case
match_materials = {'UNK-LG': 'UNKNOWN',
                   'C': 'COPPER',
                   'UNK-NOLG': 'UNKNOWN-UNLIKELY LEAD',
                   'G': 'GALVANIZED',
                   'B': 'BRASS',
                   'CI-L': 'LINED CAST IRON',
                   'CI-U': 'UNLINED CAST IRON',
                   'DI': 'DUCTILE IRON',
                   'L': 'LEAD',
                   'P': 'PLASTIC'}    

## Define verified

In [ ]:
# TODO: define here which pipes are verified
# according to customer all service lines with Known material are verified   
for df in [pub_pipes, pr_pipes]:
    df[verified_field] = False  
    cond_verify = (~df[material_field].str.contains("UNKNOWN", na=False))
    df.loc[cond_verify, verified_field] = True

## Define state: active/inactive

In [ ]:
# TODO: define here which pipes are active/inactive

for df in [pub_pipes, pr_pipes]:
    df[active_field] = True

## Join ids

In [ ]:
joined_ids = pub_pipes.copy()
joined_ids[join_public_field] = joined_ids[pipe_id_field]
joined_ids[join_private_field] = joined_ids[pipe_id_field]
joined_ids = joined_ids[[join_public_field, join_private_field]].copy()

# Checks

## Initialize

In [ ]:
# TODO: define the pipes in the project
project.pub_pipes = pub_pipes.copy()
project.pr_pipes = pr_pipes.copy()
project.abd_pub_pipes = None
project.abd_pr_pipes = None
project.joined_ids = joined_ids.copy()

In [ ]:
# TODO: set the template parameters
template = LeadTemplate(project)

if match_materials is not None and isinstance(match_materials, dict):
    template.material_dict = match_materials
    
if match_diameters is not None and isinstance(match_diameters, dict):
    template.diameter_dict = match_diameters

In [ ]:
# default parameters for the checks
template.min_cluster_distance = 5000  # ft, the maximum distance to find the isolated pipes
template.max_geocode_distance = 500  # ft, max distance between the geometry derived from address and the actual address
template.max_nearby_geometry_distance = 5  # ft, max distance between pipes having different address
template.max_connected_distance = 0.5 # ft, max distance between the public and the connected private pipe
template.min_length = 1 # ft
template.max_length = 1000 # ft

In [ ]:
# checks if all fields exist
template.initial_field_checks()

## Pipe id

In [ ]:
# checks for null pipe_ids and replace them with the keyword 'UNKNOWN_1', 'UNKNOWN_2', in order to perform the checks

# the issues are written in the sheet: 'null_pipe_id'
template.check_nulls_and_preprocess_pipe_id()

## Material

In [ ]:
# checks the material and applies the common material map in all dataframes: pipes, abd_pipes and events
template.check_and_preprocess_material()

## Diameter

In [ ]:
# checks if the diameter is string and converts it to float

# the issues are written in the sheet: 'invalid_diameter'
template.check_and_preprocess_diameter()

## Diameter conflicts

In [ ]:
# finds connected public-private pipes with diameter difference > 2 inches

# the issues are written in the sheet: 'diameter_conflicts'
template.check_diameter_conflicts()

## Installation year

In [ ]:
# checks if the installation year is null or contains invalid values

# the issues are written in the sheet: 'invalid_install_yr'
template.check_and_preprocess_install_yr()

## Abandon year

In [ ]:
# checks the abandon year if it contains null or invalid values

# the issues are written in the sheet: 'invalid_abandon_yr'
template.check_and_preprocess_abandon_yr()

## Duplicates on multiple columns

In [ ]:
# checks if the pipes contain duplicate records based on multiple columns

# the issues are written in the sheet: 'double_records'
template.check_pipes_duplicate_multiple_columns()

## Duplicate pipe_id

In [ ]:
# checks for duplicate pipe_ids

# the issues are written in the sheet: 'double_pipe_id'
template.check_pipes_duplicate_pipe_id()

## Active/Inactive conflicts

In [ ]:
# checks for conflicts in the state (active/inactive) between the connected public-private pipes

# the issues are written in the sheet: 'active_state_conflicts'
template.check_conflicts_active_state()

## Verified with unknown material

In [ ]:
# checks for pipes having unknown material but they are set to verified
template.check_verified_unknowns()

## Check conflicts between pipes and abd_pipes

In [ ]:
# check for pipes with the same pipe_id between the in-place pipes and the abandoned pipes
# conflicts in the instalation/abandon year

# the issues are written in the sheet: 'abd_year_conflicts'
template.check_conflicts_abandoned_year()

In [ ]:
# this checks for records with unknown material in the abandoned pipes

# the issues are written in the sheet: 'unknowns_in_abandoned'
template.check_for_unknowns_in_abandoned()

In [ ]:
# checks for in-place pipes that seem to be replaced because their pipe_id exists in the abandoned pipes
# but they still have material Unknown

# the issues are written in the sheet: 'unknowns_replaced'
template.check_for_unknowns_in_replaced()

## Prepare geometries

In [ ]:
state, study_center = geom_l.get_geodataframe_US_state_and_center(pub_pipes)

In [ ]:
geom_l.get_unit_of_length(pub_pipes)

In [ ]:
pub_pipes.crs

In [ ]:
# finds the invalid geometries and the duplicates after exploding the geodataframes
# the continuous pipes with same install_yr, material, pipe_diam are merged

# the invalid geometries are written in the sheet: 'invalid_geometry'
# the double pipe_ids after exploding are written in the sheet: 'double_pipe_id_after_explode'
template.check_and_preprocess_geometries()

In [ ]:
# finds invalid coordinates

# the issues are written in the sheet: 'invalid_coordinates'
template.check_and_remove_invalid_coordinates()

In [ ]:
# finds the duplicate geometries

# the issues are written in the sheet: 'duplicate_geometries'
template.check_duplicate_geometries()

In [ ]:
# get the geometries that are nearby, within a distance of max_nearby_geometry_distance, and checks if they have different address

# the issues are written in the sheet: 'nearby_geometries'
template.check_nearby_geometries()

## Pipe length

In [ ]:
# the issues are written in the sheets: 'short_pipes' and 'long_pipes'
template.check_pipe_length()

## Address

In [ ]:
# checks for null address or null street numbers

# the issues are written in the sheet: 'invalid_address'
template.check_and_preprocess_address()

## Conficts between address and geometry

In [ ]:
# 1. this checks the distance between the geometry derived from geocoding, using the address, and the actual geometry
# if it less than the max_geocode_distance

# the output must be checked on the map before giving the list with the errors to the customer

# the issues are written in the sheet: 'conflict_geocode_geometries'
template.check_address_geometry_conflicts()

In [ ]:
# 2. this checks if the address derived from the geometry (using inverse geocoding) matches the actual address
# the output addresses must be checked on the map before giving the list to the customer

# the issues are written in the sheet: 'mismatched_addresses'
template.check_for_mismatched_addresses()

## Connection issues

In [ ]:
# this check:
# 1. finds the public pipes that are not connected with any private pipe, 
# 2. finds the public pipes that are connected with multiple private pipes

# 3. finds the private pipes that are not connected with any public pipe, 
# 4. finds the private pipes that are connected with multiple public pipes

# the max distance between the public and private side is template.max_connected_distance
# the issues are written in the sheets: multiple_connections and unknown_connections

In [ ]:
template.check_connection_issues()

## Invalid installation year of Lead pipes

In [ ]:
# the issues are written in the sheet: 'lead_invalid_install_yr'
template.check_lead_with_invalid_install_yr()

## Invalid diameter of Lead pipes

In [ ]:
# the issues are written in the sheet: 'lead_invalid_diameter'
template.check_lead_with_invalid_diameter()

## Merge all pipes

In [ ]:
template.concatenate_act_and_abd_dfs()

## Statistics per material

In [ ]:
# check for invalid installation year or diameter values per material
template.pipes_statistics_per_material()

## Isolated pipes

In [ ]:
# check for pipes that are isolated, meaning that are not connected with the rest of the system pipes, 
# within a distance of template.min_cluster_distance

# the isolated piped of the active pipes are written in the sheet: 'act_isolated_pipes'
# the isolated piped of the abandoned pipes are written in the sheet: 'abd_isolated_pipes'
template.check_for_isolated_pipes()

# Final output

In [ ]:
template.save_issues_in_xlsx()